<a href="https://colab.research.google.com/github/spiritualitasalma-lgtm/anita/blob/main/Modul_Praktikum_Pipeline_Roboflow_YOLO_UAS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Modul Praktikum Berkelanjutan: Pipeline Ekstraksi Ciri Berbasis Anotasi Bounding Box Roboflow**
Modul ini dirancang khusus untuk memproses dataset yang diunduh langsung dari platform **Roboflow** (Format ekspor: **YOLO v8 / YOLO v5**).

Mahasiswa akan belajar bagaimana membaca file teks anotasi koordinat koordinat, melakukan normalisasi/denormalisasi piksel secara spasial, melakukan pemotongan objek otomatis (*Cropping ROI*), mengekstrak deskriptor **Bentuk, Warna, dan Tekstur GLCM**, serta menyusun data angka tersebut menjadi file **CSV Dataset Training** yang siap dikirim ke algoritma Machine Learning.

## **1. Persiapan Environment & Unduh Dataset dari Roboflow**
Silakan ganti kode unduhan di bawah ini menggunakan tautan link perintah ekspor (`!curl` atau `!wget`) yang Bapak peroleh dari akun Roboflow Bapak. Sebagai demonstrasi fungsional, kita akan membuat simulasi folder data dengan struktur YOLO standar agar script langsung berjalan mulus tanpa error.

In [1]:
import os
import cv2
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage import feature

!pip install roboflow
from roboflow import Roboflow
rf = Roboflow(api_key="6p6ia3ZsBgtOIGVmWIiI")
project = rf.workspace("anithas-workspace-ufg2v").project("pcp_uas-8ia64")
version = project.version(1)
dataset = version.download("yolov8")
# Removed redundant line: dataset = version.download("yolov8")

# Alternative source to avoid 429 Too Many Requests errors
image_url = "https://id.pngtree.com/freebackground/vibrant-fresh-produce-a-colorful-display-of-fruits-and-vegetables_15776935.html"
image_filename = "sayurbuahan.jpg"

# =========================================================================
# TODO UNTUK MAHASISWA: Tempelkan tautan download script dari Roboflow di sini
# Contoh:
# !pip install roboflow
# !wget -O dataset.zip "https://app.roboflow.com/ds/YOUR_LINK_HERE"
# !unzip -q dataset.zip -d dataset_roboflow/
# =========================================================================


# BENTUK SIMULASI STRUKTUR DIRECTORY ROBOFLOW (Agar script langsung fungsional saat di-run pertama kali)
# Menggunakan 'Project uas_pcp' untuk konsistensi dengan hasil download Roboflow
os.makedirs("pcp_uas-1/train/images", exist_ok=True)
os.makedirs("pcp_uas-1/train/labels", exist_ok=True)

# Buat gambar sampel buatan (Dummy Image) buah sayuraan di folder yang benar
dummy_img = np.zeros((400, 400, 3), dtype=np.uint8)
cv2.circle(dummy_img, (150, 150), 40, (20, 20, 180), -1)   # Kelas 0: Eritrosit (Merah)
cv2.circle(dummy_img, (280, 260), 55, (220, 30, 30), -1)   # Kelas 1: Leukosit (Biru di BGR)
cv2.imwrite(os.path.join("pcp_uas-1/train/images/", image_filename), dummy_img)

# Buat file anotasi koordinat YOLO terkait (Format: kelas_id x_center y_center width height) di folder yang benar
with open(os.path.join("pcp_uas-1/train/labels/", image_filename + ".txt"), "w") as f:
    f.write("0 0.375 0.375 0.20 0.20\n") # Eritrosit terpusat di (150,150)
    f.write("1 0.700 0.650 0.28 0.28\n") # Leukosit terpusat di (280,260)

print("✓ Direktori simulasi Roboflow berhasil dibentuk!")

loading Roboflow workspace...
loading Roboflow project...
✓ Direktori simulasi Roboflow berhasil dibentuk!


## **2. Pembuatan Fungsi Inti Ekstraktor Ciri Berbasis Koordinat Bounding Box**
Fungsi ini bertugas membuka satu file citra dan membaca file anotasi pendampingnya, mengonversi titik koordinat koordinat ternormalisasi YOLO menjadi koordinat piksel, memotong wilayah ROI, dan mengekstrak matriks ciri multivariat.

In [2]:
def ekstraksi_ciri_objek(image_path, label_path):
    # 1. Load citra
    img = cv2.imread(image_path)
    # Check if image was loaded successfully before accessing its shape
    if img is None:
        print(f"Error: Image not loaded from {image_path}")
        return None

    h, w, _ = img.shape

    # 2. Baca file anotasi YOLO
    if not os.path.exists(label_path):
        return None

    data_fitur = []
    with open(label_path, 'r') as f:
        lines = f.readlines()

    for line in lines:
        parts = line.split()
        class_id = int(parts[0])
        # Koordinat YOLO: x_center, y_center, width, height (ternormalisasi 0-1)
        x_c, y_c, bw, bh = map(float, parts[1:])

        # 3. Konversi ke koordinat piksel
        x_pixel = int(x_c * w)
        y_pixel = int(y_c * h)
        box_w = int(bw * w)
        box_h = int(bh * h)

        x_min = max(0, x_pixel - box_w // 2)
        y_min = max(0, y_pixel - box_h // 2)

        # 4. Memotong ROI (Region of Interest)
        roi = img[y_min : y_min + box_h, x_min : x_min + box_w]

        if roi.size == 0: continue

        # 5. Ekstraksi Ciri (Multivariat)
        # Fitur Warna: Rata-rata BGR
        mean_b, mean_g, mean_r = cv2.mean(roi)[:3]

        # Fitur Tekstur: Menggunakan GLCM (Homogeneity & Contrast)
        gray_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        glcm = feature.graycomatrix(gray_roi, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)
        homogeneity = feature.graycoprops(glcm, 'homogeneity')[0, 0]
        contrast = feature.graycoprops(glcm, 'contrast')[0, 0]

        data_fitur.append({
            "class_id": class_id,
            "mean_r": mean_r, "mean_g": mean_g, "mean_b": mean_b,
            "homogeneity": homogeneity, "contrast": contrast
        })

    return data_fitur

# --- Contoh Penggunaan ---
image_path = os.path.join("pcp_uas-1/train/images/", image_filename)
label_path = os.path.join("pcp_uas-1/train/labels/", image_filename + ".txt")

fitur_list = ekstraksi_ciri_objek(image_path, label_path)
df_features = pd.DataFrame(fitur_list)
print("✓ Berhasil mengekstrak fitur ke DataFrame:")
print(df_features.head())

✓ Berhasil mengekstrak fitur ke DataFrame:
   class_id      mean_r     mean_g      mean_b  homogeneity   contrast
0         0  141.279531  16.559062   16.585156     0.790224  72.997152
1         1   23.757334  22.965003  166.446827     0.852131  38.058961


In [3]:
import os
import pandas as pd
import cv2
import numpy as np
from skimage import feature

def ekstrak_fitur_dari_roboflow(path_gambar, path_label, peta_kelas):
    """
    Fungsi membaca koordinat YOLO hasil anotasi manusia dari Roboflow,
    melakukan crop regional objek, dan mengekstrak ciri internal.
    """
    img_asli = cv2.imread(path_gambar)
    if img_asli is None or not os.path.exists(path_label):
        return []

    H, W, _ = img_asli.shape
    img_gray = cv2.cvtColor(img_asli, cv2.COLOR_BGR2GRAY)
    b_channel = img_asli[:, :, 0] # Ambil Spektral Kanal Biru (Format BGR OpenCV)

    fitur_gambar_ini = []

    # Baca file label teks YOLO
    with open(path_label, 'r') as file:
        lines = file.readlines()

    for line in lines:
        data = line.strip().split()
        if len(data) != 5:
            continue

        id_kelas = int(data[0])
        x_center, y_center, width, height = map(float, data[1:])

        # 1. DENORMALISASI KOORDINAT YOLO (0-1) KE KOORDINAT PIKSEL CITRA
        x_min = int((x_center - width/2) * W)
        x_max = int((x_center + width/2) * W)
        y_min = int((y_center - height/2) * H)
        y_max = int((y_center + height/2) * H)

        # Proteksi pengaman koordinat agar tidak keluar dari dimensi matriks gambar
        x_min, x_max = max(0, x_min), min(W, x_max)
        y_min, y_max = max(0, y_min), min(H, y_max)

        # 2. POTONG REGIONAL OBJEK (CROP REGION OF INTEREST)
        roi_warna_b = b_channel[y_min:y_max, x_min:x_max]
        roi_gray = img_gray[y_min:y_max, x_min:x_max]

        # Antisipasi apabila hasil cropping menghasilkan wilayah kosong (Nol)
        if roi_gray.size == 0 or (y_max - y_min) < 3 or (x_max - x_min) < 3:
            continue

        # 3. PERHITUNGAN CIRI GEOMETRI EKSTERNAL (Bentuk Bounding Box)
        area_bbox = (y_max - y_min) * (x_max - x_min)
        aspect_ratio = (x_max - x_min) / (y_max - y_min)

        # 4. PERHITUNGAN CIRI SPEKTRAL WARNA INTERNAL (Warna Orde 1)
        mean_blue = np.mean(roi_warna_b)

        # 5. PERHITUNGAN CIRI TEKSTUR SPASIAL (Matriks Haralick GLCM)
        # Kuantisasi ke 16 tingkatan keabuan agar efisien dan bebas noise pecah
        roi_kuantisasi = (roi_gray / 16).astype(np.uint8)

        glcm = feature.graycomatrix(roi_kuantisasi, distances=[1], angles=[0],
                                     levels=16, symmetric=True, normed=True)
        contrast = feature.graycoprops(glcm, 'contrast')[0, 0]
        energy = feature.graycoprops(glcm, 'energy')[0, 0]

        # Ubah ID Angka kelas menjadi Label String sesuai konvensi Roboflow
        nama_kelas = peta_kelas.get(id_kelas, f"Kelas_{id_kelas}")

        # Gabungkan semua ciri ke dalam struktur baris data terpadu
        fitur_gambar_ini.append({
            'Area_BBox': area_bbox,
            'Aspect_Ratio': round(aspect_ratio, 2),
            'Mean_Blue': round(mean_blue, 2),
            'Contrast_GLCM': round(contrast, 4),
            'Energy_GLCM': round(energy, 4),
            'Label_Target': nama_kelas # Label murni hasil anotasi divalidasi pakar
        })

    return fitur_gambar_ini

# Tentukan kamus pemetaan (Mapping) kelas.
# Sesuaikan urutan ID ini dengan urutan saat Bapak membuat anotasi di platform Roboflow
peta_id_ke_nama = {
    0: "wortel",
    1: "mawar",
    2: "Manggis",
    3: "Kunyit",
    4: "bunga",
    5: "bougainvillea",
}
master_dataset_roboflow = []

# Sesuaikan dengan letak folder train hasil ekstraksi file zip Roboflow
# Menggunakan 'pcp_uas 1' untuk konsistensi dengan hasil download Roboflow
folder_images = "pcp_uas-1/train/images/"
folder_labels = "pcp_uas-1/train/labels/"

print("=== MEMULAI ALUR EKSTRAKSI MASAL BERBASIS ANOTASI ===")

# Ambil seluruh berkas gambar di folder gambar
daftar_file_gambar = os.listdir(folder_images)
counter_proses = 0

for file_name in daftar_file_gambar:
    if file_name.lower().endswith(('.jpg', '.jpeg', '.png')):
        basename = os.path.splitext(file_name)[0]

        path_img = os.path.join(folder_images, file_name)
        path_lbl = os.path.join(folder_labels, basename + ".txt")

        # Jalankan ekstraksi
        fitur_list = ekstrak_fitur_dari_roboflow(path_img, path_lbl, peta_id_ke_nama)
        master_dataset_roboflow.extend(fitur_list)
        counter_proses += 1

# Konversi akumulasi data objek masal menjadi Pandas DataFrame
df_final_roboflow = pd.DataFrame(master_dataset_roboflow)

# Simpan sebagai berkas CSV untuk modal pokok algoritma Machine Learning minggu depan
nama_csv_output = "dataset_fitur_roboflow.csv"
df_final_roboflow.to_csv(nama_csv_output, index=False)

print(f"\n✔ SELESAI: Berhasil memproses {counter_proses} file gambar.")
print(f"✔ Total objek sel yang berhasil diekstrak dan diberi label resmi: {len(df_final_roboflow)} objek.")
print(f"✔ Berkas database terstruktur sukses disimpan dalam file: '{nama_csv_output}'")

# Tampilkan sampel isi database teratas
df_final_roboflow.head()

=== MEMULAI ALUR EKSTRAKSI MASAL BERBASIS ANOTASI ===

✔ SELESAI: Berhasil memproses 1 file gambar.
✔ Total objek sel yang berhasil diekstrak dan diberi label resmi: 0 objek.
✔ Berkas database terstruktur sukses disimpan dalam file: 'dataset_fitur_roboflow.csv'


""


## **3. Automasi Iterasi Masal Dataset & Ekspor Berkas CSV**
Kita membuat perulangan utama untuk merambah folder dataset Roboflow secara masal, mengekstrak ciri setiap objek dari ratusan file gambar, dan menyatukannya ke dalam tabel terpadu Pandas DataFrame.

## **4. Lembar Diskusi Evaluasi Mahasiswa**
1. **Analisis Normalisasi Spasial:** Mengapa koordinat yang dihasilkan oleh Roboflow di file `.txt` berupa angka desimal di bawah 1 (misalnya `0.375`)? Apa keuntungan normalisasi koordinat ini bagi flexibilitas model apabila gambar asli diperkecil (*resize*)?
2. **Optimalisasi Ciri Tekstur:** Amati nilai *Contrast_GLCM* dan *Energy_GLCM* pada objek kelas Leukosit vs Eritrosit yang berhasil Anda kumpulkan di file CSV. Apakah ciri tekstur terbukti efektif memberikan pemisah kuantitatif yang signifikan antar-kelas objek?